In [142]:
import pandas as pd
import numpy as np
import pickle, json

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut, GridSearchCV
from sklearn.feature_selection import RFECV
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBRegressor
    use_xgb = True
except ImportError:
    use_xgb = False

import warnings
warnings.filterwarnings('ignore')

OUTPUT_PATH = "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/data/outputs/"

Load Data

In [143]:
## MESSAGE FOR TEAMMATES ==> CHANGE THIS ON OWN MACHINE
base_path = '~/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/Project 2 - Matchday Attendance Prediction/Data/'

df_match    = pd.read_csv(base_path + 'gold_match.csv')
df_trends   = pd.read_csv(base_path + 'gold_google_trends_daily.csv')
df_tickets  = pd.read_csv(base_path + 'gold_match_tickets.csv')
df_context  = pd.read_csv(base_path + 'gold_match_context.csv')
df_goals    = pd.read_csv(base_path + 'gold_match_goals.csv')
df_articles = pd.read_csv(base_path + 'gold_belga_press_articles.csv',
                           escapechar='\\', on_bad_lines='skip')

# Quick merged view for inspection
matches = pd.read_csv(base_path + 'gold_match.csv')
context = pd.read_csv(base_path + 'gold_match_context.csv')
tickets = pd.read_csv(base_path + 'gold_match_tickets.csv')

data = matches.merge(context, on="match_id", how="left")
data = data.merge(tickets, on="match_id", how="left")
data = data.dropna(subset=["tickets_scanned"])

print("Merged view shape:", data.shape)
data.head()

Merged view shape: (80, 70)


,match_id,match_date_x,kickoff_time_local,match_date_utc,kickoff_time_utc,matchday,competition_id,competition_name,season,season_id,...,campaign_motto,tickets_sold_b2c,tickets_sold_b2b,tickets_sold_total,seasonpass_holders,tickets_trib1,tickets_trib2_thuis,tickets_trib2_uit,tickets_trib3,tickets_trib4
1,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2022-07-30Z,16:15:00Z,2.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,NaN,488.0,764.0,5025.0,4032.0,1899.0,127.0,0.0,1427.0,1572.0
3,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,18:30:00,2022-08-14Z,16:30:00Z,4.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,NaN,1448.0,1284.0,6263.0,4032.0,2387.0,323.0,0.0,1982.0,1571.0
5,d65hmi7sq03yzr5he1k7ypus4,2022-08-27,18:15:00,2022-08-27Z,16:15:00Z,6.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,NaN,439.0,823.0,5025.0,4032.0,1876.0,126.0,0.0,1453.0,1570.0
7,d80mkemezkz16bqh6lbn8tlhw,2022-09-10,20:45:00,2022-09-10Z,18:45:00Z,8.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,NaN,782.0,906.0,5438.0,4032.0,1897.0,560.0,0.0,1411.0,1570.0
9,dak40etbhbqsr1nxyt50qcg0k,2022-10-01,16:00:00,2022-10-01Z,14:00:00Z,10.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,NaN,1115.0,970.0,5844.0,4032.0,2163.0,355.0,0.0,1757.0,1569.0


DATA PREP

In [144]:
df = df_match[df_match['is_home_match'] == True].copy()
df = df.merge(df_tickets, on='match_id', how='left')
df = df.merge(df_context, on='match_id', how='left')
df['match_date'] = df['match_date_x']
df = df.drop(columns=['match_date_x', 'match_date_y'], errors='ignore')

# Google Trends — 7-day average pre-match
df_trends_agg = df_trends.groupby('match_id')['ohl_interest'].mean().reset_index()
df = df.merge(df_trends_agg, on='match_id', how='left')

# Total article count (all linked articles)
df_articles_agg = df_articles.groupby('match_id').size().reset_index(name='article_count')
df = df.merge(df_articles_agg, on='match_id', how='left')
df['article_count'] = df['article_count'].fillna(0)

# Pre-match article count (7 days before matchday only — no leakage)
df_articles['date'] = pd.to_datetime(df_articles['date'])

art_pre = []
for _, row in df[['match_id', 'match_date']].iterrows():
    window = df_articles[
        (df_articles['match_id'] == row['match_id']) &
        (df_articles['days_to_match'].between(-7, -1))
    ]
    art_pre.append({'match_id': row['match_id'], 'article_count_7d': len(window)})

df_art_pre = pd.DataFrame(art_pre)
df = df.merge(df_art_pre, on='match_id', how='left')
df['article_count_7d'] = df['article_count_7d'].fillna(0)

df = df.sort_values('match_date').reset_index(drop=True)
print("Data prep complete:", df.shape)

Data prep complete: (71, 72)


Feature Engineering (Combined)


In [145]:
df['kickoff_hour'] = pd.to_datetime(
    df['kickoff_time_local'], format='%H:%M:%S'
).dt.hour

# ── Form features ─────────────────────────────────────────────────────────────
points_map = {'W': 3, 'D': 1, 'L': 0}
df['points'] = df['result_home'].map(points_map)
df['points_last_5'] = df['points'].rolling(5).sum().shift(1).fillna(df['points'].mean() * 5)
df['win'] = (df['result_home'] == 'W').astype(int)
df['wins_last_3'] = df['win'].rolling(3).sum().shift(1).fillna(df['win'].mean() * 3)
df['goal_diff'] = df['goals_home_ft'] - df['goals_away_ft']
df['goal_diff_last_5'] = df['goal_diff'].rolling(5).sum().shift(1).fillna(0)

# ── Season & matchday ─────────────────────────────────────────────────────────
df['matchday'] = pd.to_numeric(df['matchday'], errors='coerce')
df['season_progress'] = df['matchday'] / df['matchday'].max()

# ── Match importance composite ────────────────────────────────────────────────
df['form_strength_norm'] = (
    (df['points_last_5'] - df['points_last_5'].min()) /
    (df['points_last_5'].max() - df['points_last_5'].min())
)
df['match_importance'] = 0.5 * df['form_strength_norm'] + 0.5 * df['season_progress']
df['is_high_importance'] = (df['match_importance'] > df['match_importance'].median()).astype(int)

# ── Opponent features ─────────────────────────────────────────────────────────
df['opponent'] = df['away_team']
df['opponent_freq'] = df['opponent'].map(df['opponent'].value_counts())

top_teams = ["Club Brugge", "Anderlecht", "STVV", "KV Mechelen", "Westerlo"]
df['is_top_opponent'] = df['away_team'].isin(top_teams).astype(int)

df['opponent_strength_norm'] = (
    (df['opponent_freq'] - df['opponent_freq'].min()) /
    (df['opponent_freq'].max() - df['opponent_freq'].min())
)

# ── Match attractiveness ──────────────────────────────────────────────────────
df['match_attractiveness'] = (
    0.4 * df['match_importance'] +
    0.4 * df['opponent_strength_norm'] +
    0.2 * df['is_top_opponent']
)
df['form_x_opponent'] = df['points_last_5'] * df['is_top_opponent']

# ── Opponent avg attendance ───────────────────────────────────────────────────
df['opponent_avg_attendance_raw'] = df.groupby('away_team')['tickets_sold_total'].transform('mean')

# ── Lag & promo ───────────────────────────────────────────────────────────────
df['attendance_lag_1'] = df['tickets_scanned'].shift(1).fillna(df['tickets_scanned'].mean())
df['has_promotion'] = df['has_promotion'].astype(int)

# ── Calendar flags ────────────────────────────────────────────────────────────
df['is_school_holiday_flanders'] = df['is_school_holiday_flanders'].astype(int)
df['is_public_holiday']          = df['is_public_holiday'].astype(int)
df['is_midweek']                 = df['is_midweek'].astype(int)

# ── Play-off flag ─────────────────────────────────────────────────────────────
df['is_playoff'] = df['stage'].str.contains('Playoff|Play-off', case=False, na=False).astype(int)

# ── Last result vs opponent ───────────────────────────────────────────────────
result_map = {'W': 1, 'D': 0, 'L': -1}
df['last_result_vs_opponent_enc'] = df['last_result_vs_opponent'].map(result_map).fillna(0)

# ── Weather ───────────────────────────────────────────────────────────────────
df['ohl_interest']   = df['ohl_interest'].fillna(df['ohl_interest'].median())
df['weather_score']  = df['weather_score'].fillna(df['weather_score'].median())
df['weather_rain_mm'] = df['weather_rain_mm'].fillna(df['weather_rain_mm'].median())

# ── Interaction features ──────────────────────────────────────────────────────
df['promo_x_top_opponent'] = df['has_promotion'] * df['is_top_opponent']
df['lag_x_form']           = df['attendance_lag_1'] * df['points_last_5']
df['rain_x_weekend']       = df['weather_rain_mm'] * df['is_weekend']
df['playoff_x_top']        = df['is_playoff'] * df['is_top_opponent']
# ── Rolling attendance ────────────────────────────────────────────────────────
df['attendance_roll_3'] = df['tickets_scanned'].rolling(3).mean().shift(1).fillna(df['tickets_scanned'].mean())
df['attendance_roll_5'] = df['tickets_scanned'].rolling(5).mean().shift(1).fillna(df['tickets_scanned'].mean())

# ── Season encoding ───────────────────────────────────────────────────────────
df['season_enc'] = pd.factorize(df['season'])[0]

# ── Opponent one-hot encoding ─────────────────────────────────────────────────
opp_dummies = pd.get_dummies(df['away_team'], prefix='opp').astype(int)
df = pd.concat([df, opp_dummies], axis=1)
opp_cols = [c for c in df.columns if c.startswith('opp_')]

print("Feature engineering complete:", df.shape)



Feature engineering complete: (71, 121)


Sigmoid Normalization (Optional)


In [146]:
STADIUM_CAPACITY = 10_000
USE_SIGMOID_TARGET = False

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def normalize_attendance(y, capacity=STADIUM_CAPACITY):
    rate = y / capacity
    centered = rate - 0.5
    return sigmoid(centered * 6)

def inverse_sigmoid_attendance(y_sig, capacity=STADIUM_CAPACITY):
    clipped = np.clip(y_sig, 1e-6, 1 - 1e-6)
    centered = np.log(clipped / (1 - clipped)) / 6
    return (centered + 0.5) * capacity

Features set

In [147]:
features_combined = [

    # ── Ticketing ─────────────────────────────────────────────────────────────
    'attendance_lag_1',
    'attendance_roll_3',
    'attendance_roll_5',
    'tickets_sold_b2c',             # ✅ safe — individual pre-match sales
    'tickets_sold_b2b',             # ✅ safe — corporate pre-match bookings
    # 'tickets_sold_total',         # ❌ removed — same-day data, leakage
    'seasonpass_holders',
    # 'tickets_trib1',              # ❌ removed — derived from sold_total
    # 'tickets_trib2_thuis',        # ❌ removed — derived from sold_total
    # 'tickets_trib3',              # ❌ removed — derived from sold_total
    # 'tickets_trib4',              # ❌ removed — derived from sold_total

    # ── Opponent ──────────────────────────────────────────────────────────────
    'opponent_avg_attendance_raw',
    'is_top_opponent',
    'last_result_vs_opponent_enc',

    # ── Sporting form ─────────────────────────────────────────────────────────
    'points_last_5',
    'goal_diff_last_5',
    'form_x_opponent',

    # ── Match context ─────────────────────────────────────────────────────────
    'match_importance',
    'match_attractiveness',
    'season_progress',
    'season_enc',
    'is_playoff',

    # ── Commercial / promo ────────────────────────────────────────────────────
    'has_promotion',
    'pct_free_tickets',

    # ── Timing & calendar ─────────────────────────────────────────────────────
    'is_weekend',
    'is_midweek',
    'kickoff_hour',
    'is_school_holiday_flanders',
    'is_public_holiday',
    'academic_week',

    # ── Weather ───────────────────────────────────────────────────────────────
    'weather_score',
    'weather_rain_mm',

    # ── Media buzz ────────────────────────────────────────────────────────────
    'ohl_interest',
    'article_count_7d',

    # ── Interactions ──────────────────────────────────────────────────────────
    'promo_x_top_opponent',
    'lag_x_form',
    'rain_x_weekend',
    'playoff_x_top',

] + opp_cols                        # opponent one-hot dummies

features_combined = [col for col in features_combined if col in df.columns]

target   = 'tickets_scanned'
df_model = df[features_combined + [target, 'away_team']].dropna()
X        = df_model[features_combined]
y        = df_model[target]
y_fit    = normalize_attendance(y) if USE_SIGMOID_TARGET else y

print(f"Full feature dataset: {df_model.shape}")
print(f"Active features ({len(features_combined)}): {features_combined}")

Full feature dataset: (71, 56)
Active features (54): ['attendance_lag_1', 'attendance_roll_3', 'attendance_roll_5', 'tickets_sold_b2c', 'tickets_sold_b2b', 'seasonpass_holders', 'opponent_avg_attendance_raw', 'is_top_opponent', 'last_result_vs_opponent_enc', 'points_last_5', 'goal_diff_last_5', 'form_x_opponent', 'match_importance', 'match_attractiveness', 'season_progress', 'season_enc', 'is_playoff', 'has_promotion', 'pct_free_tickets', 'is_weekend', 'is_midweek', 'kickoff_hour', 'is_school_holiday_flanders', 'is_public_holiday', 'academic_week', 'weather_score', 'weather_rain_mm', 'ohl_interest', 'article_count_7d', 'promo_x_top_opponent', 'lag_x_form', 'rain_x_weekend', 'playoff_x_top', 'opp_Anderlecht', 'opp_Antwerp', 'opp_Beerschot', 'opp_Cercle Brugge', 'opp_Club Brugge', 'opp_Dender', 'opp_Eupen', 'opp_Genk', 'opp_Gent', 'opp_KV Oostende', 'opp_Kortrijk', 'opp_Mechelen', 'opp_RAAL La Louvière', 'opp_RWDM', 'opp_Seraing', 'opp_Sint-Truiden', 'opp_Sporting Charleroi', 'opp_Standa

Feature Importance Ranking

In [148]:
scaler_fi   = StandardScaler()
X_scaled_fi = scaler_fi.fit_transform(X)

rf_fi = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
gb_fi = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
rr_fi = Ridge(alpha=10)

rf_fi.fit(X_scaled_fi, y_fit)
gb_fi.fit(X_scaled_fi, y_fit)
rr_fi.fit(X_scaled_fi, y_fit)

fi_rf   = pd.Series(rf_fi.feature_importances_, index=features_combined, name="RandomForest")
fi_gb   = pd.Series(gb_fi.feature_importances_, index=features_combined, name="GradientBoosting")
fi_rr   = pd.Series(np.abs(rr_fi.coef_),        index=features_combined, name="Ridge_abs_coef")

perm    = permutation_importance(rf_fi, X_scaled_fi, y_fit, n_repeats=30, random_state=42)
fi_perm = pd.Series(perm.importances_mean,       index=features_combined, name="Permutation")

fi_df      = pd.concat([fi_rf, fi_gb, fi_rr, fi_perm], axis=1)
fi_df_norm = fi_df.apply(lambda col: (col - col.min()) / (col.max() - col.min()))
fi_df_norm["mean_score"] = fi_df_norm.mean(axis=1)
fi_df_norm = fi_df_norm.sort_values("mean_score", ascending=False)

print("── Feature Importance Ranking (normalised 0–1) ──────────────")
print(fi_df_norm.round(3).to_string())

── Feature Importance Ranking (normalised 0–1) ──────────────
                             RandomForest  GradientBoosting  Ridge_abs_coef  Permutation  mean_score
attendance_roll_3                   1.000             1.000           0.371        1.000       0.843
attendance_lag_1                    0.719             0.543           0.545        0.619       0.607
tickets_sold_b2c                    0.289             0.239           1.000        0.321       0.462
match_attractiveness                0.301             0.427           0.291        0.279       0.325
tickets_sold_b2b                    0.317             0.231           0.116        0.368       0.258
pct_free_tickets                    0.141             0.163           0.511        0.151       0.241
opponent_avg_attendance_raw         0.223             0.237           0.315        0.180       0.238
goal_diff_last_5                    0.179             0.202           0.358        0.146       0.221
season_progress              

RFECV Feature Selection

In [149]:
PROTECTED = [
    'attendance_lag_1',
    'attendance_roll_3',
    'tickets_sold_b2c',
    'tickets_sold_b2b',
    'opponent_avg_attendance_raw',
    'weather_rain_mm',
    'has_promotion',
    'points_last_5',
    'is_top_opponent',
]

rfe = RFECV(
    estimator=Ridge(alpha=100),
    step=1,
    cv=LeaveOneOut(),
    scoring='neg_mean_absolute_error',
    min_features_to_select=3
)

unprotected = [f for f in features_combined if f not in PROTECTED]
rfe.fit(StandardScaler().fit_transform(df_model[unprotected]), y)

rfecv_selected  = [f for f, s in zip(unprotected, rfe.support_) if s]
eliminated      = [f for f, s in zip(unprotected, rfe.support_) if not s]
selected_features = [f for f in features_combined if f in rfecv_selected + PROTECTED]

print(f"🔒 Protected:            {PROTECTED}")
print(f"✅ RFECV selected ({len(rfecv_selected)}): {rfecv_selected}")
print(f"❌ Eliminated ({len(eliminated)}):    {eliminated}")
print(f"\n✅ Final feature set ({len(selected_features)}):")
for f in selected_features:
    print(f"   + {f}")

mae_per_n = -rfe.cv_results_['mean_test_score']
n_range   = range(rfe.min_features_to_select, len(unprotected) + 1)
print(f"\n── MAE by feature count (excl. protected) ───────────────────")
for n, mae in zip(n_range, mae_per_n):
    marker = " ← optimal" if n == rfe.n_features_ else ""
    print(f"  n={n:>2}  MAE: {mae:>8.1f}{marker}")

🔒 Protected:            ['attendance_lag_1', 'attendance_roll_3', 'tickets_sold_b2c', 'tickets_sold_b2b', 'opponent_avg_attendance_raw', 'weather_rain_mm', 'has_promotion', 'points_last_5', 'is_top_opponent']
✅ RFECV selected (20): ['attendance_roll_5', 'goal_diff_last_5', 'form_x_opponent', 'match_importance', 'match_attractiveness', 'season_progress', 'season_enc', 'is_playoff', 'kickoff_hour', 'ohl_interest', 'article_count_7d', 'lag_x_form', 'opp_Anderlecht', 'opp_Antwerp', 'opp_Club Brugge', 'opp_Dender', 'opp_KV Oostende', 'opp_Sporting Charleroi', 'opp_Standard Liège', 'opp_Westerlo']
❌ Eliminated (25):    ['seasonpass_holders', 'last_result_vs_opponent_enc', 'pct_free_tickets', 'is_weekend', 'is_midweek', 'is_school_holiday_flanders', 'is_public_holiday', 'academic_week', 'weather_score', 'promo_x_top_opponent', 'rain_x_weekend', 'playoff_x_top', 'opp_Beerschot', 'opp_Cercle Brugge', 'opp_Eupen', 'opp_Genk', 'opp_Gent', 'opp_Kortrijk', 'opp_Mechelen', 'opp_RAAL La Louvière', 'o

Apply Selected Features


In [150]:
features_combined = selected_features

df_model = df[features_combined + [target, 'away_team']].dropna()
X        = df_model[features_combined]
y        = df_model[target]
y_fit    = normalize_attendance(y) if USE_SIGMOID_TARGET else y

print(f"✅ Final dataset: {df_model.shape}")
print(f"✅ Features: {features_combined}")

✅ Final dataset: (71, 31)
✅ Features: ['attendance_lag_1', 'attendance_roll_3', 'attendance_roll_5', 'tickets_sold_b2c', 'tickets_sold_b2b', 'opponent_avg_attendance_raw', 'is_top_opponent', 'points_last_5', 'goal_diff_last_5', 'form_x_opponent', 'match_importance', 'match_attractiveness', 'season_progress', 'season_enc', 'is_playoff', 'has_promotion', 'kickoff_hour', 'weather_rain_mm', 'ohl_interest', 'article_count_7d', 'lag_x_form', 'opp_Anderlecht', 'opp_Antwerp', 'opp_Club Brugge', 'opp_Dender', 'opp_KV Oostende', 'opp_Sporting Charleroi', 'opp_Standard Liège', 'opp_Westerlo']


Hyperparameter Tuning

In [151]:
def tune_model(name, model, param_grid, X_scaled, y):
    gs = GridSearchCV(model, param_grid, cv=5, scoring='r2', n_jobs=-1)
    gs.fit(X_scaled, y)
    print(f"  [{name}] Best params: {gs.best_params_}  |  CV R²: {gs.best_score_:.3f}")
    return gs.best_estimator_

scaler_tune   = StandardScaler()
X_scaled_tune = scaler_tune.fit_transform(X)

best_ridge = tune_model("Ridge", Ridge(), {
    'alpha': [0.1, 1, 10, 50, 100, 200, 500]
}, X_scaled_tune, y)

best_rf = tune_model("RandomForest", RandomForestRegressor(random_state=42), {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4],
    'min_samples_leaf': [3, 5, 8],
    'max_features': [0.5, 0.7, 1.0]
}, X_scaled_tune, y)

best_gb = tune_model("GradientBoosting", GradientBoostingRegressor(random_state=42), {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'min_samples_leaf': [3, 5, 8]
}, X_scaled_tune, y)

if use_xgb:
    best_xgb = tune_model("XGBoost", XGBRegressor(random_state=42), {
        'n_estimators': [50, 100, 200],
        'max_depth': [2, 3],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.7, 0.8, 1.0],
        'reg_lambda': [1, 5, 10],
        'min_child_weight': [3, 5, 8]
    }, X_scaled_tune, y)

print("\n✅ All models tuned.")

  [Ridge] Best params: {'alpha': 10}  |  CV R²: 0.158
  [RandomForest] Best params: {'max_depth': 4, 'max_features': 0.5, 'min_samples_leaf': 3, 'n_estimators': 300}  |  CV R²: -0.582
  [GradientBoosting] Best params: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 8, 'n_estimators': 100, 'subsample': 0.7}  |  CV R²: -0.114
  [XGBoost] Best params: {'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 5, 'n_estimators': 100, 'reg_lambda': 10, 'subsample': 0.7}  |  CV R²: -0.299

✅ All models tuned.


Model Training + LOOCV

In [152]:
stacking_model = StackingRegressor(
    estimators=[
        ('ridge', best_ridge),
        ('rf',    best_rf),
        ('gb',    best_gb),
    ],
    final_estimator=Ridge(alpha=10),
    cv=5
)

models = {
    "LinearRegression": LinearRegression(),
    "Ridge":            best_ridge,
    "RandomForest":     best_rf,
    "GradientBoosting": best_gb,
    "Stacking":         stacking_model,
}
if use_xgb:
    models["XGBoost"] = best_xgb

loo = LeaveOneOut()
results, predictions_dict = {}, {}

for name, model in models.items():
    y_true_list, y_pred_list = [], []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train_fold    = y_fit.iloc[train_idx]

        if 'opponent_avg_attendance_raw' in features_combined:
            opp_avg         = df_model.iloc[train_idx].groupby('away_team')[target].mean()
            global_avg_fold = y.iloc[train_idx].mean()
            opp_test        = df_model.iloc[test_idx]['away_team'].values[0]
            X_train = X_train.copy(); X_test = X_test.copy()
            X_train['opponent_avg_attendance_raw'] = df_model.iloc[train_idx]['away_team'].map(opp_avg)
            X_test['opponent_avg_attendance_raw']  = opp_avg.get(opp_test, global_avg_fold)

        scaler  = StandardScaler()
        X_tr_s  = scaler.fit_transform(X_train)
        X_te_s  = scaler.transform(X_test)

        model.fit(X_tr_s, y_train_fold)
        pred = model.predict(X_te_s)[0]

        if USE_SIGMOID_TARGET:
            pred = inverse_sigmoid_attendance(pred)

        y_true_list.append(y.iloc[test_idx].values[0])
        y_pred_list.append(pred)

    results[name] = {
        "MAE":  mean_absolute_error(y_true_list, y_pred_list),
        "RMSE": np.sqrt(mean_squared_error(y_true_list, y_pred_list)),
        "R2":   r2_score(y_true_list, y_pred_list),
        "MAPE": np.mean(np.abs((np.array(y_true_list) - np.array(y_pred_list)) / np.array(y_true_list))) * 100
    }
    predictions_dict[name] = pd.DataFrame({"Actual": y_true_list, "Predicted": y_pred_list})

results_df = pd.DataFrame(results).T.sort_values("MAE")
print(results_df.round(2))

                      MAE     RMSE    R2   MAPE
Ridge              979.85  1234.27  0.61  15.70
XGBoost           1003.00  1296.73  0.57  16.39
GradientBoosting  1010.72  1295.37  0.57  16.46
RandomForest      1058.49  1314.00  0.56  17.17
LinearRegression  1063.98  1333.71  0.55  17.20
Stacking          1071.48  1305.00  0.57  17.13


Overfit Check

In [153]:
train_results = {}

for name, model in models.items():
    scaler_full = StandardScaler()
    X_full_s    = scaler_full.fit_transform(X)
    model.fit(X_full_s, y_fit)
    y_train_pred = model.predict(X_full_s)

    if USE_SIGMOID_TARGET:
        y_train_pred = inverse_sigmoid_attendance(y_train_pred)

    train_results[name] = {
        "Train_MAE": mean_absolute_error(y, y_train_pred),
        "Train_R2":  r2_score(y, y_train_pred),
    }

train_df   = pd.DataFrame(train_results).T
overfit_df = results_df[["MAE", "R2"]].join(train_df)
overfit_df["MAE_gap"] = overfit_df["MAE"] - overfit_df["Train_MAE"]
overfit_df["R2_gap"]  = overfit_df["R2"]  - overfit_df["Train_R2"]

print("── Overfit Check (LOOCV vs Full-Train) ──────────────────────")
print(overfit_df[["Train_MAE", "MAE", "MAE_gap", "Train_R2", "R2", "R2_gap"]].round(3))

print("\n── Overfit Flag (MAE_gap > 500 or R2_gap < -0.3) ───────────")
flags = overfit_df[(overfit_df["MAE_gap"] > 500) | (overfit_df["R2_gap"] < -0.3)]
if flags.empty:
    print("✅ No severe overfitting detected.")
else:
    print("⚠️  Potential overfitting in:")
    print(flags[["MAE_gap", "R2_gap"]].round(3))

# ── Model selection ───────────────────────────────────────────────────────────
MAX_ACCEPTABLE_GAP = 500

clean_models    = overfit_df[
    (overfit_df["MAE_gap"] <= MAX_ACCEPTABLE_GAP) &
    (overfit_df["R2_gap"]  >= -0.3)
]

if clean_models.empty:
    print("\n⚠️  All models exceed thresholds — picking lowest MAE_gap")
    best_model_name = overfit_df["MAE_gap"].idxmin()
else:
    best_model_name = clean_models["MAE"].idxmin()

print(f"\n{'─'*54}")
print(f"  🏆 Selected model : {best_model_name}")
print(f"  📉 LOOCV MAE      : {overfit_df.loc[best_model_name, 'MAE']:.1f} tickets")
print(f"  📈 LOOCV R²       : {overfit_df.loc[best_model_name, 'R2']:.3f}")
print(f"  🔒 MAE gap        : {overfit_df.loc[best_model_name, 'MAE_gap']:.1f}")
print(f"  🔒 R² gap         : {overfit_df.loc[best_model_name, 'R2_gap']:.3f}")
print(f"{'─'*54}")
print(f"  ℹ️  Selected for lowest LOOCV MAE among non-overfitting models")

── Overfit Check (LOOCV vs Full-Train) ──────────────────────
                  Train_MAE       MAE  MAE_gap  Train_R2     R2  R2_gap
Ridge               647.084   979.852  332.768     0.824  0.612  -0.212
XGBoost             335.591  1003.004  667.413     0.948  0.572  -0.376
GradientBoosting    173.147  1010.723  837.576     0.987  0.573  -0.414
RandomForest        598.421  1058.492  460.070     0.852  0.560  -0.292
LinearRegression    624.217  1063.976  439.759     0.842  0.547  -0.294
Stacking            490.735  1071.481  580.746     0.909  0.566  -0.342

── Overfit Flag (MAE_gap > 500 or R2_gap < -0.3) ───────────
⚠️  Potential overfitting in:
                  MAE_gap  R2_gap
XGBoost           667.413  -0.376
GradientBoosting  837.576  -0.414
Stacking          580.746  -0.342

──────────────────────────────────────────────────────
  🏆 Selected model : Ridge
  📉 LOOCV MAE      : 979.9 tickets
  📈 LOOCV R²       : 0.612
  🔒 MAE gap        : 332.8
  🔒 R² gap         : -0.212
──────

SAVE BEST MODEL


In [154]:
MAX_ACCEPTABLE_GAP = 600
generalising = overfit_df[overfit_df["MAE_gap"] <= MAX_ACCEPTABLE_GAP]

if generalising.empty:
    print("⚠️ All models overfit — picking lowest MAE_gap")
    best_model_name = overfit_df["MAE_gap"].idxmin()
else:
    best_model_name = generalising["MAE"].idxmin()

print(f"\n✅ Best model: {best_model_name}")

scaler_final   = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)
best_model     = models[best_model_name]
best_model.fit(X_scaled_final, y_fit)

std_resid = (y_fit - pd.Series(best_model.predict(X_scaled_final))).std()

with open('/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/notebooks/interactive_interface/model.pkl', 'wb') as f:
    pickle.dump({
        'model':    best_model,
        'scaler':   scaler_final,
        'std':      float(std_resid),
        'features': features_combined,
        'sigmoid':  USE_SIGMOID_TARGET,
        'capacity': STADIUM_CAPACITY
    }, f)

opp_lookup = df_model.groupby('away_team')[target].mean().round(0).to_dict()
opp_lookup['__global_avg__'] = float(y.mean())
with open('/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/notebooks/interactive_interface/opponent_lookup.json', 'w') as f:
    json.dump(opp_lookup, f, indent=2)

results_df.to_csv(OUTPUT_PATH + "model_results_combined.csv")
print("✅ model.pkl, opponent_lookup.json, model_results_combined.csv saved.")


✅ Best model: Ridge
✅ model.pkl, opponent_lookup.json, model_results_combined.csv saved.
